# NONAN candidate healthy-cohort materialization

Materialise only the predeclared 80-person `candidate_healthy_enrichment` partition from the completed NONAN GaitPrint downloads. This is a data-readiness step, **not** a model-training, threshold-selection, calibration, or external-evaluation step. The 29-person frozen healthy-specificity cohort, three structural-audit people, and 14 mobility-screen exclusions are asserted disjoint before any signal is read.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
NONAN = PROJECT_ROOT / 'data' / 'interim' / 'nonan_gaitprint'
PACKAGES = PROJECT_ROOT / 'data' / 'raw' / 'nonan_gaitprint' / 'source_packages' / 'candidate_healthy_enrichment'
partitions = pd.read_csv(NONAN / 'participant_partitions.csv')
candidate = set(partitions.loc[partitions.partition.eq('candidate_healthy_enrichment'), 'participant_id'])
frozen = set(partitions.loc[partitions.partition.eq('frozen_healthy_specificity'), 'participant_id'])
structural = set(partitions.loc[partitions.partition.eq('structural_audit_only'), 'participant_id'])
excluded = set(partitions.loc[partitions.partition.eq('excluded_mobility_screen'), 'participant_id'])
archives = {path.stem for path in PACKAGES.glob('S*.zip')}
manifest = json.loads((NONAN / 'download_manifest_candidate_healthy_enrichment.json').read_text(encoding='utf-8'))
verified = {item['participant_id'] for item in manifest if item['status'] in {'verified_existing', 'downloaded_verified'}}

assert len(candidate) == 80
assert candidate == archives == verified
assert not (candidate & frozen)
assert not (candidate & structural)
assert not (candidate & excluded)
print({'candidate_participants': len(candidate), 'verified_archives': len(verified), 'frozen_overlap': 0, 'structural_overlap': 0, 'mobility_exclusion_overlap': 0})
partitions.loc[partitions.partition.eq('candidate_healthy_enrichment')].groupby(['cohort', 'gender']).size().rename('participants').reset_index()

{'candidate_participants': 80, 'verified_archives': 80, 'frozen_overlap': 0, 'structural_overlap': 0, 'mobility_exclusion_overlap': 0}


,cohort,gender,participants
0,middle,female,18
1,middle,male,12
2,older,female,12
3,older,male,13
4,young,female,12
5,young,male,13


## Locked conversion contract

Each archived CSV is streamed directly from its ZIP package, avoiding a full extraction that would exceed the available disk space. The converter uses the documented lower-spine L1/T12 proxy plus left/right foot triaxial acceleration, converts mG to g, applies only isolated interior spike repair for a sensitivity representation, resamples 200 Hz to 100 Hz, converts each sensor to an acceleration magnitude, and writes non-overlapping 5-second windows with channel order `LB/LF/RF`. No candidate-specific normalisation is fitted here.

In [2]:
command = [sys.executable, str(PROJECT_ROOT / 'scripts' / 'materialize_nonan_staged_audit.py'), '--partition', 'candidate_healthy_enrichment']
completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True, check=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

{"partition": "candidate_healthy_enrichment", "windows": [68398, 500, 3], "patched_samples": 178, "raw_max_g": 100.33024597167969, "repaired_max_g": 15.741238594055176}



In [3]:
prefix = 'candidate_healthy_enrichment'
raw = np.load(NONAN / f'{prefix}_magnitude_raw.npy', mmap_mode='r')
repaired = np.load(NONAN / f'{prefix}_magnitude_isolated_spike_repaired.npy', mmap_mode='r')
metadata = pd.read_csv(NONAN / f'{prefix}_window_metadata.csv')
summary = json.loads((NONAN / f'{prefix}_materialization.json').read_text(encoding='utf-8'))

assert raw.shape == repaired.shape
assert raw.shape[1:] == (500, 3)
assert len(metadata) == raw.shape[0]
assert set(metadata['participant']) == candidate
assert set(metadata['participant_key']) == {f'nonan_gaitprint:{participant}' for participant in candidate}
assert metadata['label'].eq('healthy').all()
assert metadata['partition'].eq('candidate_healthy_enrichment').all()
assert np.isfinite(raw).all() and np.isfinite(repaired).all()
assert not (set(metadata['participant']) & (frozen | structural | excluded))

participant_windows = metadata.groupby('participant').size()
print({'shape': tuple(raw.shape), 'participants': int(metadata['participant'].nunique()), 'windows': int(len(metadata)), 'windows_per_participant_min': int(participant_windows.min()), 'windows_per_participant_median': float(participant_windows.median()), 'windows_per_participant_max': int(participant_windows.max()), 'patched_samples': summary['patched_samples']})
metadata.groupby(['cohort'] if 'cohort' in metadata.columns else ['partition']).size()

{'shape': (68398, 500, 3), 'participants': 80, 'windows': 68398, 'windows_per_participant_min': 432, 'windows_per_participant_median': 864.0, 'windows_per_participant_max': 864, 'patched_samples': 178}


partition
candidate_healthy_enrichment    68398
dtype: int64

## Result boundary and next gate

The materialised candidate cohort is now structurally compatible with the three-channel tensor shape, but it is still healthy-only and uses an L1/T12 lower-spine proxy rather than documented L5. It must not be naively pooled with the binary development cohort. The next stage is a source-aware, participant-grouped compatibility and bounded-enrichment experiment that: (1) fits preprocessing only inside development folds, (2) preserves the frozen NONAN and RevalExo cohorts, and (3) accepts the candidate source only if it does not degrade original-source validation or held-out candidate-healthy specificity.